# Actual RAG: Research Paper Q&A
This notebook downloads a real research paper ("Attention Is All You Need"), builds a FAISS-based RAG pipeline on top of it, and then tests it autonomously using our Multi-Agent QA framework.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import requests
from dotenv import load_dotenv

# Load API keys from .env file
load_dotenv()

# Download the "Attention Is All You Need" paper if we don't have it
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
pdf_path = "attention_is_all_you_need.pdf"

if not os.path.exists(pdf_path):
    print("Downloading research paper...")
    response = requests.get(pdf_url)
    with open(pdf_path, "wb") as f:
        f.write(response.content)
    print("Downloaded!")
else:
    print("Paper already downloaded.")

Paper already downloaded.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

print("1. Loading document...")
loader = PyPDFLoader(pdf_path)
docs = loader.load()

print("2. Chunking document...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print("3. Building Vector Store (FAISS)...")
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("4. Creating QA Chain...")
llm = ChatOpenAI(model="gpt-4o-mini")
system_prompt = (
    "You are an assistant for question-answering tasks based on a research paper. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

def ask_research_paper(query: str) -> dict:
    """Return answer plus retrieved context so QA agents can inspect grounding."""
    response = rag_chain.invoke({"input": query})
    contexts = [doc.page_content for doc in response.get("context", [])]
    return {
        "output": response.get("answer", ""),
        "contexts": contexts,
        "retrieved_contexts": contexts,
    }

print("\nRAG System Ready. Let's test it manually:")
manual_response = ask_research_paper("What is self-attention?")
print("Q: What is self-attention?")
print("A:", manual_response["output"])
print(f"Retrieved contexts: {len(manual_response['contexts'])}")


### Autonomous Testing Phase
Now we pass the RAG function to `agentic_qa`. The wrapper returns both the answer and retrieved contexts, so the upgraded Judge can attach local quality, retrieval inspection, hallucination grounding labels, expanded RAGAS metrics, security flags, and regression-memory replays.


In [ ]:
import os
from agentic_qa.sut import CallableAdapter, set_active_sut
from agentic_qa.graph.workflow import run_qa_pipeline
from agentic_qa import display_qa_results

# Configure testing parameters for a quick test run.
os.environ["MAX_ITERATIONS"] = "2"
os.environ["TESTS_PER_ITERATION"] = "3"
os.environ.setdefault("REGRESSION_MEMORY_REPLAY", "3")

adapter = CallableAdapter(
    fn=ask_research_paper,
    system_name="Research Paper RAG (Attention Is All You Need)",
    description="A Retrieval-Augmented Generation system that answers questions about the 'Attention Is All You Need' machine learning research paper.",
    domain="Academic Research / Machine Learning",
)
set_active_sut(adapter)

print("Launching Autonomous Multi-Agent QA...\n")
final_state = run_qa_pipeline()

verdicts_df, regression_df = display_qa_results(final_state)


In [ ]:
# (Replaced by display_qa_results in the cell above)


In [ ]:
# (Replaced by display_qa_results in the cell above)
